# 📊 Benchmark Phase 1 — MaroTrade Intelligence v4
### Pipeline 100% data-driven · Reproductible · Format PFE

> **Le score ML (data-driven) détermine le classement.
> Les 7 dimensions sont une lecture métier post-hoc pour l'explicabilité PME,
> pas la cible d'entraînement.**

| Paramètre | Valeur |
|---|---|
| **Source** | `data/raw/comtrade_ag2_full.csv` + accords/WB/OCDE/Trends |
| **Périmètre** | ~18 pays accords · ~85 produits AG2 · 2018–2024 |
| **Filtres** | value ≥ 100K USD · couple ≥ 500K cumulé & ≥ 3 ans |
| **Cible** | `log_return = log(V_{t+1}/V_t) × 100`, clip[-100, 100] |
| **Split** | Train ≤2021 · Val 2022-23 · Test 2024 |
| **CV** | Walk-forward par année — aucune fuite temporelle |
| **Sélection best model** | **Spearman val groupé** par (hs_code, year) |
| **INTERDIT** | Entraîner sur score_7D, 60/40 ou tout barème |

## Imports & Seeds

In [3]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import scipy.stats as stats
from scipy.stats import spearmanr
import json
import os
import time

# ── Seeds ─────────────────────────────────────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Style publication ─────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    '#F8F9FA',
    'axes.grid':         True,
    'grid.alpha':        0.35,
    'grid.linestyle':    '--',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'axes.titlesize':    12,
    'axes.labelsize':    10,
    'xtick.labelsize':   9,
    'ytick.labelsize':   9,
})

COLORS = {
    'Ridge':    '#534AB7',
    'XGBoost':  '#1D9E75',
    'LightGBM': '#378ADD',
    'CatBoost': '#E24B4A',
}

FEAT_LABELS = {
    'log_value_usd':      'log(Valeur export)',
    'log_lag1':           'log(Export N-1)',
    'log_ma3':            'log(Moy. mobile 3 ans)',
    'std3':               'Volatilité 3 ans',
    'growth_lag1':        'Croissance lag1 (%)',
    'log_return_lag1':    'Log-return lag1',
    'price_usd_kg':       'Prix USD/kg',
    'market_share':       'Part de marché',
    'cagr_3y':            'CAGR 3 ans (feature)',
    'accord_score':       'Score accord (ALE/PREF/NPF)',
    'droits':             'Droits de douane (%)',
    'wb_gdp_per_capita':  'PIB/habitant (WB)',
    'wb_imports_pct_gdp': 'Imports % PIB (WB)',
    'wb_available':       'WB disponible (flag)',
    'ocde_risk_score':    'Score risque OCDE',
    'distance_km':        'Distance Maroc→pays (km)',
    'trend_score':        'Score Google Trends',
}

# ── Chemins ───────────────────────────────────────────────────────────────────
DATA_DIR     = '../data/raw'
FIGURES_DIR  = '../figures'
ARTIFACTS_DIR= '../artifacts'

os.makedirs(FIGURES_DIR,   exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print('✅ Imports OK')
print(f'   NumPy   : {np.__version__}')
print(f'   Pandas  : {pd.__version__}')
print(f'   Dossiers créés : {FIGURES_DIR} · {ARTIFACTS_DIR}')

✅ Imports OK
   NumPy   : 2.4.3
   Pandas  : 2.3.3
   Dossiers créés : ../figures · ../artifacts


 ## Chargement des données brutes

In [5]:
# ── Chargement des fichiers sources ──────────────────────────────────────────
df_train = pd.read_csv(f'{DATA_DIR}/ml_train.csv')
df_val   = pd.read_csv(f'{DATA_DIR}/ml_val.csv')
df_test  = pd.read_csv(f'{DATA_DIR}/ml_test.csv')
df_acc   = pd.read_csv(f'{DATA_DIR}/accords_maroc.csv',       index_col=0)
df_wb    = pd.read_csv(f'{DATA_DIR}/worldbank_indicators.csv', index_col=0)
df_ocde  = pd.read_csv(f'{DATA_DIR}/ocde_risk.csv',            index_col=0)

with open(f'{DATA_DIR}/google_trends.json') as f:
    trends_raw = json.load(f)

# ── Vue d'ensemble ────────────────────────────────────────────────────────────
df_raw = pd.concat([df_train, df_val, df_test], ignore_index=True)

print('Fichiers chargés')
print(f'\n── Comtrade brut ──')
print(f'   Lignes    : {len(df_raw):,}')
print(f'   Colonnes  : {list(df_raw.columns)}')
print(f'   Années    : {sorted(df_raw["year"].unique())}')
print(f'   HS codes  : {df_raw["hs_code"].nunique()}')
print(f'   Pays      : {df_raw["partner_code"].nunique()}')
print(f'   partner_code dtype  : {df_raw["partner_code"].dtype}')
print(f'   partner_code sample : {list(df_raw["partner_code"].unique()[:6])}')

print(f'\n── Accords Maroc ──')
print(f'   Pays   : {len(df_acc)}')
print(f'   Index  : {list(df_acc.index[:5])}')
print(f'   Cols   : {list(df_acc.columns)}')

print(f'\n── World Bank ──')
print(f'   Pays   : {len(df_wb)}')
print(f'   Cols   : {list(df_wb.columns)}')

print(f'\n── OCDE Risk ──')
print(f'   Pays   : {len(df_ocde)}')
print(f'   Cols   : {list(df_ocde.columns)}')

print(f'\n── Google Trends ──')
print(f'   Produits : {list(trends_raw.keys())}')
geo_sample = list(list(trends_raw.values())[0].keys())
print(f'   Pays     : {geo_sample}')

print(f'\n── log_return stats (brut) ──')
lr = df_raw['log_return'].dropna()
print(f'   n    : {len(lr):,}')
print(f'   mean : {lr.mean():.2f}%')
print(f'   std  : {lr.std():.2f}%')
print(f'   min  : {lr.min():.2f}%')
print(f'   max  : {lr.max():.2f}%')

Fichiers chargés

── Comtrade brut ──
   Lignes    : 37,314
   Colonnes  : ['hs_code', 'hs_desc', 'year', 'partner_code', 'partner_name', 'value_usd', 'weight_kg', 'price_usd_kg', 'value_next', 'log_return', 'lag1', 'lag2', 'lag3', 'ma3', 'std3', 'growth_lag1', 'log_return_lag1', 'price_lag1']
   Années    : [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
   HS codes  : 93
   Pays      : 191
   partner_code dtype  : int64
   partner_code sample : [np.int64(24), np.int64(56), np.int64(124), np.int64(251), np.int64(324), np.int64(392)]

── Accords Maroc ──
   Pays   : 20
   Index  : ['FRA', 'DEU', 'USA', 'ESP', 'ITA']
   Cols   : ['accord', 'droits', 'type']

── World Bank ──
   Pays   : 20
   Cols   : ['name', 'gdp_per_capita', 'gdp_per_capita_year', 'imports_pct_gdp', 'imports_pct_gdp_year', 'trade_pct_gdp', 'trade_pct_gdp_year']

── OCDE Risk ──
   Pays   : 20
   Cols   : [

## Mappings & Constantes
Définit tous les mappings nécessaires pour relier les 3 formats de codes pays (UN numérique, ISO3, ISO2) et les clés Google Trends. Vérifie aussi la couverture réelle des pays accords dans le dataset.
Ce qu'on veut confirmer :

        Combien des 20 pays accords sont présents dans le dataset
        Si tous les pays df_acc sont mappables vers des codes UN
        Quelles colonnes numériques sont disponibles dans df_wb

In [7]:
# ── Mapping ISO3 → code numérique UN ─────────────────────────────────────────
ISO3_TO_UN = {
    'FRA': 251, 'DEU': 276, 'USA': 842, 'ESP': 724, 'ITA': 380,
    'NLD': 528, 'BEL':  56, 'GBR': 826, 'CAN': 124, 'JPN': 392,
    'SAU': 682, 'ARE': 784, 'QAT': 634, 'KWT': 414, 'CHN': 156,
    'KOR': 410, 'SGP': 702, 'AUS':  36, 'NOR': 578, 'CHE': 756,
}
UN_TO_ISO3  = {v: k for k, v in ISO3_TO_UN.items()}
ISO3_TO_ISO2 = {
    'FRA': 'FR', 'DEU': 'DE', 'USA': 'US', 'ESP': 'ES', 'ITA': 'IT',
    'NLD': 'NL', 'BEL': 'BE', 'GBR': 'GB', 'CAN': 'CA', 'JPN': 'JP',
    'SAU': 'SA', 'ARE': 'AE', 'QAT': 'QA', 'KWT': 'KW', 'CHN': 'CN',
    'KOR': 'KR', 'SGP': 'SG', 'AUS': 'AU', 'NOR': 'NO', 'CHE': 'CH',
}

# ── Distance Maroc (Casablanca) → pays partenaires (km) ──────────────────────
DISTANCE_KM = {
    251: 2400, 276: 2900, 842: 7200, 724:  750, 380: 2200,
    528: 2700,  56: 2800, 826: 2600, 124: 7800, 392: 11000,
    682: 4500, 784: 5800, 634: 5200, 414: 5100, 156:  9500,
    410: 10800, 702: 10800,  36: 16000, 578: 3800, 756: 2600,
}

# ── Pays avec accords (codes UN) ──────────────────────────────────────────────
PAYS_ACCORDS_UN = set(ISO3_TO_UN.values())

# ── Mapping HS AG2 → clés Google Trends ──────────────────────────────────────
HS_TO_TRENDS = {
    '15': ['151590', '150910'],
    '9':  ['09102010', '090920'],
    '16': ['160413'],
    '8':  ['080410', '080521'],
    '57': ['570110'],
    '69': ['691010'],
    '7':  ['070200'],
}

# ── Vérifications ─────────────────────────────────────────────────────────────
pays_dans_data = set(df_raw['partner_code'].unique())
pays_avec_acc  = PAYS_ACCORDS_UN & pays_dans_data
pays_manquants = PAYS_ACCORDS_UN - pays_dans_data

print('✅ Mappings définis')
print(f'\n── Couverture pays accords ──')
print(f'   Pays accords définis      : {len(PAYS_ACCORDS_UN)}')
print(f'   Présents dans le dataset  : {len(pays_avec_acc)}')
print(f'   Absents du dataset        : {len(pays_manquants)}')
if pays_manquants:
    absents_iso3 = [UN_TO_ISO3[p] for p in pays_manquants]
    print(f'   ISO3 absents              : {absents_iso3}')

print(f'\n── Cohérence accords ──')
acc_index_un = {ISO3_TO_UN[i] for i in df_acc.index if i in ISO3_TO_UN}
print(f'   df_acc mappable vers UN   : {len(acc_index_un)} pays')
print(f'   Exemple accord ──')
print(df_acc.head(3))

print(f'\n── Cohérence WB ──')
wb_index_un = {ISO3_TO_UN[i] for i in df_wb.index if i in ISO3_TO_UN}
print(f'   df_wb mappable vers UN    : {len(wb_index_un)} pays')
print(f'   Colonnes numériques       : {[c for c in df_wb.columns if "year" not in c and c != "name"]}')

✅ Mappings définis

── Couverture pays accords ──
   Pays accords définis      : 20
   Présents dans le dataset  : 18
   Absents du dataset        : 2
   ISO3 absents              : ['NOR', 'CHE']

── Cohérence accords ──
   df_acc mappable vers UN   : 20 pays
   Exemple accord ──
                    accord  droits type
FRA  Accord association UE     0.0  ALE
DEU  Accord association UE     0.0  ALE
USA   Accord libre-échange     0.0  ALE

── Cohérence WB ──
   df_wb mappable vers UN    : 20 pays
   Colonnes numériques       : ['gdp_per_capita', 'imports_pct_gdp', 'trade_pct_gdp']


---
## Filtres & Périmètre

In [8]:
# ── Étape 1 : Supprimer leakage ───────────────────────────────────────────────
COLS_LEAKAGE = ['value_next', 'price_lag1']
df = df_raw.drop(columns=[c for c in COLS_LEAKAGE if c in df_raw.columns])
print(f'✅ Leakage supprimé : {COLS_LEAKAGE}')

# ── Étape 2 : Filtre pays avec accords ───────────────────────────────────────
n0 = len(df)
df = df[df['partner_code'].isin(PAYS_ACCORDS_UN)].copy()
n1 = len(df)
print(f'\n── Filtre pays accords ──')
print(f'   Avant : {n0:,}  →  Après : {n1:,}  (−{n0-n1:,})')
print(f'   Pays restants : {df["partner_code"].nunique()}')

# ── Étape 3 : Filtre value_usd ≥ 100K ────────────────────────────────────────
n1b = len(df)
df  = df[df['value_usd'] >= 100_000].copy()
n2  = len(df)
print(f'\n── Filtre value_usd ≥ 100K USD ──')
print(f'   Avant : {n1b:,}  →  Après : {n2:,}  (−{n1b-n2:,})')
pct_bruit = (n1b - n2) / n1b * 100
print(f'   Lignes bruit supprimées : {pct_bruit:.1f}%')

# ── Étape 4 : Filtre couple ≥ 500K cumulé ET ≥ 3 ans ─────────────────────────
cum = df.groupby(['hs_code', 'partner_code'])['value_usd'].sum()
yrs = df.groupby(['hs_code', 'partner_code'])['year'].nunique()

couples_ok = set(zip(
    cum[cum >= 500_000].reset_index()['hs_code'],
    cum[cum >= 500_000].reset_index()['partner_code']
)) & set(zip(
    yrs[yrs >= 3].reset_index()['hs_code'],
    yrs[yrs >= 3].reset_index()['partner_code']
))

n2b = len(df)
df  = df[df.apply(
    lambda r: (r['hs_code'], r['partner_code']) in couples_ok, axis=1
)].copy()
n3  = len(df)
print(f'\n── Filtre couple ≥ 500K cumulé & ≥ 3 ans ──')
print(f'   Avant : {n2b:,}  →  Après : {n3:,}  (−{n2b-n3:,})')
print(f'   Couples valides : {len(couples_ok)}')

# ── Étape 5 : Clip log_return [-100, 100] ────────────────────────────────────
df['log_return'] = df['log_return'].clip(-100, 100)
lr_after = df['log_return'].dropna()
print(f'\n── Clip log_return [-100, 100] ──')
print(f'   std avant : 78.49%')
print(f'   std après : {lr_after.std():.2f}%')

# ── Étape 6 : Supprimer lags incomplets ──────────────────────────────────────
n3b = len(df)
df  = df.dropna(subset=['lag1', 'lag2', 'lag3', 'log_return']).copy()
n4  = len(df)
print(f'\n── Drop lags incomplets (lag1/2/3 + log_return) ──')
print(f'   Avant : {n3b:,}  →  Après : {n4:,}  (−{n3b-n4:,})')

# ── Résumé ────────────────────────────────────────────────────────────────────
print(f'\n{"="*45}')
print(f'  DATASET FINAL')
print(f'{"="*45}')
print(f'  Lignes    : {len(df):,}')
print(f'  HS codes  : {df["hs_code"].nunique()}')
print(f'  Pays      : {df["partner_code"].nunique()}')
print(f'  Années    : {sorted(df["year"].unique())}')
print(f'  log_return mean : {df["log_return"].mean():.2f}%')
print(f'  log_return std  : {df["log_return"].std():.2f}%')
print(f'{"="*45}')

✅ Leakage supprimé : ['value_next', 'price_lag1']

── Filtre pays accords ──
   Avant : 37,314  →  Après : 10,115  (−27,199)
   Pays restants : 18

── Filtre value_usd ≥ 100K USD ──
   Avant : 10,115  →  Après : 6,788  (−3,327)
   Lignes bruit supprimées : 32.9%

── Filtre couple ≥ 500K cumulé & ≥ 3 ans ──
   Avant : 6,788  →  Après : 6,460  (−328)
   Couples valides : 718

── Clip log_return [-100, 100] ──
   std avant : 78.49%
   std après : 49.88%

── Drop lags incomplets (lag1/2/3 + log_return) ──
   Avant : 6,460  →  Après : 4,673  (−1,787)

  DATASET FINAL
  Lignes    : 4,673
  HS codes  : 85
  Pays      : 18
  Années    : [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
  log_return mean : -2.78%
  log_return std  : 50.46%


##  Feature Engineering

In [10]:
# ── Copie de travail ──────────────────────────────────────────────────────────
df_fe = df.copy()

# ── Log-transforms (réduire l'effet des valeurs extrêmes) ────────────────────
df_fe['log_value_usd'] = np.log1p(df_fe['value_usd'])
df_fe['log_lag1']      = np.log1p(df_fe['lag1'].clip(lower=0))
df_fe['log_ma3']       = np.log1p(df_fe['ma3'].clip(lower=0))

# ── Part de marché (position relative dans le produit × année) ───────────────
total_hs_yr            = df_fe.groupby(['hs_code', 'year'])['value_usd'].transform('sum')
df_fe['market_share']  = df_fe['value_usd'] / total_hs_yr.replace(0, np.nan)

# ── CAGR 3 ans (feature temporelle, pas cible) ────────────────────────────────
df_fe['cagr_3y'] = np.where(
    (df_fe['lag3'] > 0) & (df_fe['value_usd'] > 0),
    (np.power(
        df_fe['value_usd'] / df_fe['lag3'].replace(0, np.nan),
        1/3
    ) - 1) * 100,
    np.nan
)

# ── Enrichissement Accords Maroc ──────────────────────────────────────────────
type_map   = {'ALE': 1.0, 'PREF': 0.6, 'NPF': 0.3}
acc_score  = {ISO3_TO_UN[i]: type_map.get(r['type'], 0.3)
              for i, r in df_acc.iterrows() if i in ISO3_TO_UN}
acc_droits = {ISO3_TO_UN[i]: float(r['droits'])
              for i, r in df_acc.iterrows() if i in ISO3_TO_UN}

df_fe['accord_score'] = df_fe['partner_code'].map(acc_score)
df_fe['droits']       = df_fe['partner_code'].map(acc_droits)

# ── Enrichissement World Bank ─────────────────────────────────────────────────
wb_gdp = {ISO3_TO_UN[i]: float(r['gdp_per_capita'])
          for i, r in df_wb.iterrows()
          if i in ISO3_TO_UN and pd.notna(r.get('gdp_per_capita'))}
wb_imp = {ISO3_TO_UN[i]: float(r['imports_pct_gdp'])
          for i, r in df_wb.iterrows()
          if i in ISO3_TO_UN and pd.notna(r.get('imports_pct_gdp'))}

df_fe['wb_gdp_per_capita']  = df_fe['partner_code'].map(wb_gdp)
df_fe['wb_imports_pct_gdp'] = df_fe['partner_code'].map(wb_imp)
df_fe['wb_available']       = df_fe['wb_gdp_per_capita'].notna().astype(int)

# ── Enrichissement OCDE ───────────────────────────────────────────────────────
ocde_sc = {ISO3_TO_UN[i]: float(r['score'])
           for i, r in df_ocde.iterrows()
           if i in ISO3_TO_UN and pd.notna(r.get('score'))}

df_fe['ocde_risk_score'] = df_fe['partner_code'].map(ocde_sc)

# ── Distance ─────────────────────────────────────────────────────────────────
df_fe['distance_km'] = df_fe['partner_code'].map(DISTANCE_KM)

# ── Google Trends ─────────────────────────────────────────────────────────────
def get_trend_score(row):
    ag2  = str(int(row['hs_code']))
    iso3 = UN_TO_ISO3.get(int(row['partner_code']))
    geo  = ISO3_TO_ISO2.get(iso3, '') if iso3 else ''
    for hs_key in HS_TO_TRENDS.get(ag2, []):
        if hs_key in trends_raw and geo in trends_raw[hs_key]:
            return float(trends_raw[hs_key][geo]['mean'])
    return 50.0  # valeur neutre si pas de données

df_fe['trend_score'] = df_fe.apply(get_trend_score, axis=1)

# ── Imputation médiane pour valeurs manquantes ────────────────────────────────
COLS_IMPUTE = [
    'wb_gdp_per_capita', 'wb_imports_pct_gdp',
    'ocde_risk_score', 'distance_km',
    'accord_score', 'droits',
    'cagr_3y', 'market_share'
]
for col in COLS_IMPUTE:
    med = df_fe[col].median()
    df_fe[col] = df_fe[col].fillna(med if pd.notna(med) else 0)

# ── Liste finale des features ─────────────────────────────────────────────────
FEATURES = [
    # Temporelles
    'log_value_usd', 'log_lag1', 'log_ma3', 'std3',
    'growth_lag1', 'log_return_lag1', 'price_usd_kg',
    'market_share', 'cagr_3y',
    # Accords
    'accord_score', 'droits',
    # World Bank
    'wb_gdp_per_capita', 'wb_imports_pct_gdp', 'wb_available',
    # Risque & Logistique
    'ocde_risk_score', 'distance_km',
    # Tendance
    'trend_score',
]
TARGET = 'log_return'

# ── Vérification ──────────────────────────────────────────────────────────────
print('✅ Feature engineering terminé')
print(f'\n── Features ({len(FEATURES)}) ──')
for f in FEATURES:
    n_miss = df_fe[f].isna().sum()
    flag   = ' ⚠️' if n_miss > 0 else ''
    print(f'   {f:<25} manquants={n_miss}{flag}')

print(f'\n── Vérification anti-leakage ──')
for col in ['value_next', 'price_lag1']:
    present = col in df_fe.columns
    print(f'   {col:<15} : {"⚠️  PRÉSENT" if present else "✅ absent"}')

print(f'\n── wb_available ──')
print(f'   Pays avec WB : {df_fe["wb_available"].sum()} / {len(df_fe)} lignes')

✅ Feature engineering terminé

── Features (17) ──
   log_value_usd             manquants=0
   log_lag1                  manquants=0
   log_ma3                   manquants=0
   std3                      manquants=0
   growth_lag1               manquants=0
   log_return_lag1           manquants=0
   price_usd_kg              manquants=0
   market_share              manquants=0
   cagr_3y                   manquants=0
   accord_score              manquants=0
   droits                    manquants=0
   wb_gdp_per_capita         manquants=0
   wb_imports_pct_gdp        manquants=0
   wb_available              manquants=0
   ocde_risk_score           manquants=0
   distance_km               manquants=0
   trend_score               manquants=0

── Vérification anti-leakage ──
   value_next      : ✅ absent
   price_lag1      : ✅ absent

── wb_available ──
   Pays avec WB : 4673 / 4673 lignes


## 6 Split temporel

In [11]:
# ── Split temporel strict ─────────────────────────────────────────────────────
train = df_fe[df_fe['year'] <= 2021].copy()
val   = df_fe[df_fe['year'].isin([2022, 2023])].copy()
test  = df_fe[df_fe['year'] == 2024].copy()

print('✅ Split temporel')
print(f'\n   Train (≤2021)    : {len(train):,} lignes | années {sorted(train["year"].unique())}')
print(f'   Val   (2022-23)  : {len(val):,} lignes | années {sorted(val["year"].unique())}')
print(f'   Test  (2024)     : {len(test):,} lignes | années {sorted(test["year"].unique())}')

# ── Préparation X / y ─────────────────────────────────────────────────────────
def prepare_xy(df_in, features, target, fit_scaler=None):
    """
    Prépare X et y depuis un DataFrame.
    - Impute les manquants par la médiane du df_in (train only si fit_scaler=None)
    - Applique RobustScaler (fit sur train, transform sur val/test)
    Retourne : X_scaled, y, hs_codes, years, scaler
    """
    from sklearn.preprocessing import RobustScaler

    d = df_in[features + [target, 'hs_code', 'year']].copy()

    # Imputation médiane — toujours sur les données courantes
    for col in features:
        d[col] = pd.to_numeric(d[col], errors='coerce')
        med    = d[col].median()
        d[col] = d[col].fillna(med if pd.notna(med) else 0)

    d = d.dropna(subset=[target])

    X         = d[features].values
    y         = d[target].values
    hs_codes  = d['hs_code'].values
    years     = d['year'].values

    if fit_scaler is None:
        # Train : fit + transform
        scaler = RobustScaler()
        X_sc   = scaler.fit_transform(X)
        return X_sc, y, hs_codes, years, scaler
    else:
        # Val / Test : transform uniquement
        X_sc = fit_scaler.transform(X)
        return X_sc, y, hs_codes, years, fit_scaler

# ── Appliquer ─────────────────────────────────────────────────────────────────
X_tr, y_tr, hs_tr, yr_tr, scaler = prepare_xy(train, FEATURES, TARGET)
X_v,  y_v,  hs_v,  yr_v,  _     = prepare_xy(val,   FEATURES, TARGET, scaler)
X_te, y_te, hs_te, yr_te, _     = prepare_xy(test,  FEATURES, TARGET, scaler)

print(f'\n── Shapes après scaling ──')
print(f'   X_tr : {X_tr.shape}  |  y_tr : {y_tr.shape}')
print(f'   X_v  : {X_v.shape}   |  y_v  : {y_v.shape}')
print(f'   X_te : {X_te.shape}  |  y_te : {y_te.shape}')

print(f'\n── y stats ──')
print(f'   y_tr  mean={y_tr.mean():.2f}%  std={y_tr.std():.2f}%')
print(f'   y_v   mean={y_v.mean():.2f}%   std={y_v.std():.2f}%')
print(f'   y_te  mean={y_te.mean():.2f}%  std={y_te.std():.2f}%')

print(f'\n── RobustScaler ──')
print(f'   Fitted sur train uniquement ✅')
print(f'   Transform appliqué sur val/test ✅')

✅ Split temporel

   Train (≤2021)    : 2,375 lignes | années [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
   Val   (2022-23)  : 1,225 lignes | années [np.int64(2022), np.int64(2023)]
   Test  (2024)     : 1,073 lignes | années [np.int64(2024)]

── Shapes après scaling ──
   X_tr : (2375, 17)  |  y_tr : (2375,)
   X_v  : (1225, 17)   |  y_v  : (1225,)
   X_te : (1073, 17)  |  y_te : (1073,)

── y stats ──
   y_tr  mean=1.03%  std=50.20%
   y_v   mean=15.33%   std=51.12%
   y_te  mean=-31.87%  std=35.67%

── RobustScaler ──
   Fitted sur train uniquement ✅
   Transform appliqué sur val/test ✅


## 7 Fonction Spearman groupé & Baselines

In [12]:
from sklearn.metrics import mean_squared_error, r2_score

# ── Fonction Spearman groupé par (hs_code, year) ─────────────────────────────
def spearman_grouped(y_true, y_pred, hs_codes, years, min_group=3):
    """
    Calcule le Spearman moyen par groupe (hs_code, year).
    Pour chaque groupe : Spearman entre y_true et y_pred sur les pays.
    C'est la métrique pertinente pour le ranking PME.
    """
    df_tmp = pd.DataFrame({
        'y':  y_true,
        'yp': y_pred,
        'hs': hs_codes,
        'yr': years
    })
    corrs = []
    for (hs, yr), grp in df_tmp.groupby(['hs', 'yr']):
        if len(grp) >= min_group:
            sp = spearmanr(grp['y'], grp['yp']).correlation
            if not np.isnan(sp):
                corrs.append(sp)
    return float(np.mean(corrs)) if corrs else 0.0

def eval_model(y_true, y_pred, hs_codes, years, label=''):
    """Calcule RMSE, R², Spearman groupé."""
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2   = float(r2_score(y_true, y_pred))
    sp   = spearman_grouped(y_true, y_pred, hs_codes, years)
    if label:
        print(f'   {label:<18} RMSE={rmse:.3f}  R²={r2:.4f}  Spearman={sp:.4f}')
    return rmse, r2, sp

# ── Baselines ─────────────────────────────────────────────────────────────────
print('── Baselines (val set) ──')

# Naïf-0 : prédit toujours 0
y_naive0 = np.zeros(len(y_v))
rmse_n0, r2_n0, sp_n0 = eval_model(y_v, y_naive0, hs_v, yr_v, 'Naïf-0')

# Naïf-lag : prédit log_return_lag1
lag_idx       = FEATURES.index('log_return_lag1')
y_naive_lag   = X_v[:, lag_idx]
rmse_nl, r2_nl, sp_nl = eval_model(y_v, y_naive_lag, hs_v, yr_v, 'Naïf-lag')

# Volume-only : Ridge sur log_value_usd seul
from sklearn.linear_model import Ridge
vol_idx    = FEATURES.index('log_value_usd')
X_vol_tr   = X_tr[:, [vol_idx]]
X_vol_v    = X_v[:,  [vol_idx]]
vol_model  = Ridge(alpha=1.0)
vol_model.fit(X_vol_tr, y_tr)
y_vol_pred = vol_model.predict(X_vol_v)
rmse_vl, r2_vl, sp_vl = eval_model(y_v, y_vol_pred, hs_v, yr_v, 'Volume-only')

BASELINES = {
    'Naif-0':      {'val_rmse': rmse_n0, 'val_r2': r2_n0, 'val_spearman': sp_n0},
    'Naif-lag':    {'val_rmse': rmse_nl, 'val_r2': r2_nl, 'val_spearman': sp_nl},
    'Volume-only': {'val_rmse': rmse_vl, 'val_r2': r2_vl, 'val_spearman': sp_vl},
}

print(f'\n── Interprétation baselines ──')
print(f'   Un modèle ML doit dépasser Naïf-0 sur Spearman')
print(f'   Spearman Naïf-0 = {sp_n0:.4f} → seuil à battre')
print(f'   Spearman Volume = {sp_vl:.4f} → signal volume seul')

── Baselines (val set) ──
   Naïf-0             RMSE=53.367  R²=-0.0899  Spearman=0.0000
   Naïf-lag           RMSE=53.453  R²=-0.0934  Spearman=-0.0535
   Volume-only        RMSE=53.121  R²=-0.0798  Spearman=-0.0859

── Interprétation baselines ──
   Un modèle ML doit dépasser Naïf-0 sur Spearman
   Spearman Naïf-0 = 0.0000 → seuil à battre
   Spearman Volume = -0.0859 → signal volume seul


## 8. SHAP — Explainability

In [ ]:
shap_imp  = np.array(ar['shap_imp'])
shap_vals = np.array(ar['shap_vals'])
X_shap    = np.array(ar['X_shap'])

idx_s   = np.argsort(shap_imp)[::-1]
top_n   = min(12, len(FEATURES))
top_idx = idx_s[:top_n]
top_lbl = [FEAT_LABELS.get(FEATURES[i], FEATURES[i]) for i in top_idx]

fig = plt.figure(figsize=(18, 10))
fig.suptitle('Figure 7 — Analyse SHAP · Explainability LightGBM · MaroTrade v4',
             fontsize=13, y=1.01, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# 7a — Bar SHAP
ax1 = fig.add_subplot(gs[0, 0])
cmap_c = plt.cm.RdYlGn(np.linspace(0.2, 0.8, top_n))[::-1]
bars_s = ax1.barh(range(top_n), shap_imp[top_idx][::-1],
                  color=cmap_c[::-1], alpha=0.85)
ax1.set_yticks(range(top_n))
ax1.set_yticklabels(top_lbl[::-1], fontsize=9)
ax1.set_xlabel('Importance SHAP |φ| moyenne')
ax1.set_title('Top features (SHAP)')
for bar, val in zip(bars_s, shap_imp[top_idx][::-1]):
    ax1.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
             f'{val:.2f}', va='center', fontsize=8)

# 7b — Beeswarm
ax2 = fig.add_subplot(gs[0, 1])
top5 = idx_s[:5]
for yi, fi in enumerate(top5):
    sv = shap_vals[:, fi]
    xv = X_shap[:, fi]
    sc = ax2.scatter(sv, [yi+np.random.uniform(-0.2,0.2) for _ in sv],
                     c=xv, cmap='RdBu_r', alpha=0.6, s=25, vmin=0, vmax=1)
ax2.set_yticks(range(5))
ax2.set_yticklabels([FEAT_LABELS.get(FEATURES[i],FEATURES[i]) for i in top5], fontsize=9)
ax2.axvline(0, color='black', lw=1, linestyle='--')
ax2.set_xlabel('Valeur SHAP'); ax2.set_title("Beeswarm — direction d'impact")
plt.colorbar(sc, ax=ax2, label='Feature value (norm.)', shrink=0.7)

# 7c — Waterfall 1 exemple
ax3 = fig.add_subplot(gs[0, 2])
wf     = shap_vals[0]
base_v = float(np.mean(ar['y_val']))
wf_idx = np.argsort(np.abs(wf))[::-1][:8]
wf_s   = wf[wf_idx]
wf_f   = [FEAT_LABELS.get(FEATURES[i], FEATURES[i]) for i in wf_idx]
cumsum = [base_v]
for s in wf_s: cumsum.append(cumsum[-1]+s)
bc_wf  = ['#1D9E75' if s>0 else '#E24B4A' for s in wf_s]
ax3.barh([0],[base_v], color='#888780', alpha=0.7, height=0.6, label=f'Base={base_v:.1f}%')
for pos,(s,c,fl) in enumerate(zip(wf_s,bc_wf,wf_f),1):
    ax3.barh([pos],[s],left=[cumsum[pos-1]],color=c,alpha=0.8,height=0.6)
    sign='+' if s>=0 else ''
    ax3.text(cumsum[pos-1]+s+(0.3 if s>=0 else -0.3),pos,
             f'{sign}{s:.1f}',va='center',ha='left' if s>=0 else 'right',fontsize=7.5)
    ax3.text(-2,pos,fl[:16],va='center',ha='right',fontsize=7.5)
ax3.axvline(cumsum[-1],color='purple',lw=2,linestyle='--',
            label=f'Pred={cumsum[-1]:.1f}%')
ax3.set_yticks([0]+list(range(1,len(wf_s)+1)))
ax3.set_yticklabels(['Base']+[f'F{i+1}' for i in range(len(wf_s))],fontsize=8)
ax3.set_xlabel('Impact (%)'); ax3.set_title("Waterfall — 1 prédiction exemple")
ax3.legend(fontsize=8)

# 7d — Scatter val complet
ax4 = fig.add_subplot(gs[1,:])
yp_b = np.array(ar.get(f'y_pred_val_{best}', ar['y_val']))
lim  = max(abs(y_val).max(), abs(yp_b).max())*0.8
sc4  = ax4.scatter(y_val, yp_b, alpha=0.1, s=4,
                   c=np.abs(resid), cmap='RdYlGn_r', vmin=0, vmax=40)
ax4.plot([-lim,lim],[-lim,lim],'k--',lw=2,label='Parfait')
z2   = np.polyfit(y_val, yp_b, 1)
xr4  = np.linspace(-lim,lim,100)
ax4.plot(xr4,np.poly1d(z2)(xr4),'r-',lw=2,label=f'Reg(slope={z2[0]:.3f})')
sp_g = spearmanr(y_val,yp_b).correlation
ax4.text(0.02,0.97,
         f'Val RMSE={ar["results"][best]["val_rmse"]:.2f}\n'
         f'Val R²={ar["results"][best]["val_r2"]:.4f}\n'
         f'Spearman ρ={ar["results"][best]["val_spearman"]:.4f}',
         transform=ax4.transAxes,va='top',fontsize=10,
         bbox=dict(boxstyle='round',facecolor='white',
                   edgecolor=COLORS.get(best,'#534AB7'),alpha=0.85))
ax4.set_xlim(-lim,lim); ax4.set_ylim(-lim,lim)
ax4.set_xlabel('Log-Return réel (%)'); ax4.set_ylabel('Log-Return prédit (%)')
ax4.set_title(f'Prédictions vs Réalité · {best} · Val set 2022-23')
ax4.legend(fontsize=9); plt.colorbar(sc4,ax=ax4,label='|Résidu|',shrink=0.4)

plt.savefig('figures/fig7_shap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Figure 7 | Top SHAP: {FEAT_LABELS.get(FEATURES[idx_s[0]], FEATURES[idx_s[0]])}')